# Network IDS - Modular Deep Learning Pipeline

This notebook uses the refactored `src/` modules required by `SPEC.md`: dataset adapters, leakage-safe preprocessing, imbalance analysis, ML baselines, deep-learning training, metrics, and artifact export.


## 1. Colab Setup

Run this cell on Google Colab. If your project is stored in a different Google Drive path, update `PROJECT_ROOT`.

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

import os
import sys
from pathlib import Path

PROJECT_ROOT = Path('/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path.cwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

print('Project root:', PROJECT_ROOT)
print('Files:', sorted([p.name for p in PROJECT_ROOT.iterdir()])[:20])


In [ ]:
# Colab usually has the heavy packages. Install only small project dependencies if missing.
import importlib.util
import subprocess
import sys

missing = []
for package, import_name in [('pyyaml', 'yaml'), ('joblib', 'joblib')]:
    if importlib.util.find_spec(import_name) is None:
        missing.append(package)

if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

print('Dependency check complete')


## Global Data Paths

Run this cell once after mounting Drive. It defines project and CICIDS2017 paths used by diagnostic cells and preprocessing.


In [ ]:
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd()

raw_dir = PROJECT_ROOT / 'data/raw/cicids2017'
cache_path = PROJECT_ROOT / 'data/cache/cicids2017/merged.csv'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('CICIDS2017 raw_dir:', raw_dir)
print('CICIDS2017 cache_path:', cache_path)
print('raw_dir exists:', raw_dir.exists())


## Source Package Sanity Check

Run this cell before importing project modules. It clears stale Python import caches and verifies that the required source files on Drive contain the expected functions/classes.


In [ ]:
import importlib
import shutil
import sys
from pathlib import Path

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd()

src_root = PROJECT_ROOT / 'src'
print('Source root:', src_root)
print('Source root exists:', src_root.exists())

for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

for cache_dir in src_root.rglob('__pycache__'):
    shutil.rmtree(cache_dir, ignore_errors=True)

importlib.invalidate_caches()

required_symbols = {
    'adapters/__init__.py': ['CICIDS2017Adapter', 'NSLKDDAdapter', 'UNSWNB15Adapter'],
    'evaluation/metrics.py': ['def compute_classification_metrics', 'def false_alarm_rate'],
    'imbalance/__init__.py': ['analyze_class_distribution', 'compute_balanced_class_weights', 'select_strategies'],
    'models/__init__.py': ['CNN1D', 'MLP'],
    'training/__init__.py': ['TorchTrainer'],
    'utils/seed.py': ['def set_seed'],
}

missing = []
for rel_path, symbols in required_symbols.items():
    file_path = src_root / rel_path
    if not file_path.exists():
        missing.append(f'Missing file: {file_path}')
        continue
    text = file_path.read_text(encoding='utf-8', errors='replace')
    for symbol in symbols:
        if symbol not in text:
            missing.append(f'Missing symbol `{symbol}` in {file_path}')

if missing:
    print('Source package is not updated correctly:')
    for item in missing:
        print('-', item)
    raise RuntimeError('Upload/sync the latest local src/ folder to Google Drive, then restart runtime.')

print('Source package sanity check passed')


## 2. Imports and Experiment Configuration

In [ ]:
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset

# Clear stale project imports before importing from src.
import importlib
import sys
for name in list(sys.modules):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]
importlib.invalidate_caches()

from src.adapters import CICIDS2017Adapter, NSLKDDAdapter, UNSWNB15Adapter
from src.evaluation.metrics import compute_classification_metrics
from src.imbalance import analyze_class_distribution, compute_balanced_class_weights, select_strategies
from src.models import CNN1D, MLP
from src.models.ml_baselines import build_logistic_regression, build_random_forest
from src.training import TorchTrainer
from src.utils.seed import set_seed

CFG = {
    'seed': 42,
    'dataset': 'cicids2017',       # nsl_kdd | unsw_nb15 | cicids2017
    'classification': 'multi',  # multi | binary
    'batch_size': 512,
    'epochs': 10,
    'run_ml': False,   # Baseline results already exist; avoid accidental full CICIDS2017 rerun.
    'run_dl': True,
    'dl_models': ['MLP'],       # Add 'CNN1D' for a heavier deep-learning run.
}

set_seed(CFG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
print('Project imports: OK')


## CICIDS2017 File Diagnostics

Run this cell before preprocessing when `CFG['dataset'] = 'cicids2017'`. It checks whether the CSV files are in the expected folder, which encodings can read them, and whether an old cache should be deleted.


In [ ]:
from pathlib import Path
import pandas as pd

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd()

raw_dir = PROJECT_ROOT / 'data/raw/cicids2017'
cache_path = PROJECT_ROOT / 'data/cache/cicids2017/merged.csv'

print('Project root:', PROJECT_ROOT)
print('Raw directory:', raw_dir)
print('Raw directory exists:', raw_dir.exists())
files = sorted(raw_dir.glob('*.csv')) if raw_dir.exists() else []
print('CSV files found:', len(files))
for file in files:
    print('-', file.name, f'{file.stat().st_size / (1024 * 1024):.2f} MB')

if cache_path.exists():
    print('Existing cache:', cache_path, f'{cache_path.stat().st_size / (1024 * 1024):.2f} MB')
    print('Delete the cache if it was created before the latest adapter fixes:')
    print('cache_path.unlink()')
else:
    print('No merged cache exists yet.')

encodings = ['utf-8', 'utf-8-sig', 'cp1252', 'latin1']
for file in files[:10]:
    print('\nChecking:', file.name)
    for encoding in encodings:
        try:
            sample = pd.read_csv(file, encoding=encoding, nrows=5, low_memory=False)
            print('  OK', encoding, '| columns:', list(sample.columns)[:5], '| rows:', len(sample))
            print('  label-like:', [c for c in sample.columns if 'label' in c.lower()])
            break
        except Exception as exc:
            print('  FAIL', encoding, '|', type(exc).__name__, str(exc)[:120])


## CICIDS2017 Local Runtime Copy

Google Drive streaming can be slow or unstable with large CICIDS2017 CSV files. Run this cell before preprocessing to copy the visible CSV files from Drive to Colab local disk, then point the adapter to the local copy.


In [ ]:
from pathlib import Path
import shutil

try:
    PROJECT_ROOT
except NameError:
    PROJECT_ROOT = Path('/content/drive/MyDrive/Nids_deep_learning/ids_deep_learning')
    if not PROJECT_ROOT.exists():
        PROJECT_ROOT = Path.cwd()

DRIVE_CICIDS_RAW = PROJECT_ROOT / 'data/raw/cicids2017'
LOCAL_CICIDS_RAW = Path('/content/cicids2017_raw')
LOCAL_CICIDS_CACHE = Path('/content/cicids2017_cache/merged.csv')

LOCAL_CICIDS_RAW.mkdir(parents=True, exist_ok=True)
LOCAL_CICIDS_CACHE.parent.mkdir(parents=True, exist_ok=True)

source_files = sorted(DRIVE_CICIDS_RAW.glob('*.csv')) if DRIVE_CICIDS_RAW.exists() else []
print('Drive raw path:', DRIVE_CICIDS_RAW)
print('Drive CSV files visible to Colab:', len(source_files))

if not source_files:
    raise FileNotFoundError(f'No CICIDS2017 CSV files found in {DRIVE_CICIDS_RAW}')

for src in source_files:
    dst = LOCAL_CICIDS_RAW / src.name
    if not dst.exists() or dst.stat().st_size != src.stat().st_size:
        shutil.copy2(src, dst)
    print('-', dst.name, f'{dst.stat().st_size / (1024 * 1024):.2f} MB')

if LOCAL_CICIDS_CACHE.exists():
    LOCAL_CICIDS_CACHE.unlink()
    print('Deleted local CICIDS2017 cache:', LOCAL_CICIDS_CACHE)

raw_dir = LOCAL_CICIDS_RAW
cache_path = LOCAL_CICIDS_CACHE
print('Local raw path:', raw_dir)
print('Local cache path:', cache_path)


## 3. Dataset Adapters and Leakage-Safe Preprocessing

The required order is: load raw/cache data, clean data, map labels, split train/validation/test, fit the scaler on train only, then transform validation and test.

In [ ]:
ADAPTERS = {
    'nsl_kdd': lambda: NSLKDDAdapter(
        cache_path=PROJECT_ROOT / 'data/cache/nsl_kdd/KDDTrain+.txt',
        remote_url='https://raw.githubusercontent.com/defcom17/NSL_KDD/master/KDDTrain+.txt',
        seed=CFG['seed'],
    ),
    'unsw_nb15': lambda: UNSWNB15Adapter(
        cache_path=PROJECT_ROOT / 'data/cache/unsw_nb15/training-set.csv',
        remote_url='https://raw.githubusercontent.com/ushukkla/nospammers/master/UNSW_NB15_training-set.csv',
        seed=CFG['seed'],
    ),
    'cicids2017': lambda: CICIDS2017Adapter(
        raw_dir=Path('/content/cicids2017_raw') if Path('/content/cicids2017_raw').exists() else PROJECT_ROOT / 'data/raw/cicids2017',
        cache_path=Path('/content/cicids2017_cache/merged.csv') if Path('/content/cicids2017_raw').exists() else PROJECT_ROOT / 'data/cache/cicids2017/merged.csv',
        remote_url=None,
        seed=CFG['seed'],
    ),
}

adapter = ADAPTERS[CFG['dataset']]()
output = adapter.preprocess(classification_type=CFG['classification'])

print('Dataset:', output.metadata['dataset'])
print('Features:', len(output.feature_names))
print('Labels:', output.label_mapping)
print('Train/Val/Test:', output.X_train.shape, output.X_val.shape, output.X_test.shape)
print('Cleaning:', output.metadata['cleaning'])


In [ ]:
artifact_dir = PROJECT_ROOT / 'artifacts' / CFG['dataset'] / CFG['classification']
artifact_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(output.scaler, artifact_dir / 'scaler.pkl')
joblib.dump(output.encoders, artifact_dir / 'encoders.pkl')
(artifact_dir / 'label_mapping.json').write_text(json.dumps(output.label_mapping, indent=2), encoding='utf-8')
(artifact_dir / 'inference_config.json').write_text(json.dumps({
    'dataset': CFG['dataset'],
    'classification': CFG['classification'],
    'feature_names': output.feature_names,
    'scaler': 'scaler.pkl',
    'encoders': 'encoders.pkl',
    'label_mapping': 'label_mapping.json',
}, indent=2), encoding='utf-8')

print('Saved preprocessing artifacts to:', artifact_dir)


## 4. Imbalance Analysis

In [ ]:
class_names = [None] * len(output.label_mapping)
for name, idx in output.label_mapping.items():
    class_names[idx] = name

imbalance_report = analyze_class_distribution(
    output.y_train,
    class_names=class_names,
    output_path=PROJECT_ROOT / 'results' / f"{CFG['dataset']}_{CFG['classification']}_imbalance.json",
)
strategies = select_strategies(imbalance_report)

pd.DataFrame(imbalance_report['classes']).assign(strategy=lambda df: df['label'].map(strategies))


## CICIDS2017 Focal Loss + Weighted Sampler Experiment

Use this focused rerun after preprocessing and imbalance analysis. It keeps Colab as the training environment, but calls the reusable project runner instead of duplicating training logic in the notebook.


In [ ]:
RUN_CICIDS2017_IMBALANCE_EXPERIMENTS = True

# Run one or two experiments at a time on free Colab. Keep sample_fraction=0.1
# until a setting has acceptable FAR, then scale to 0.25, 0.5, and finally 1.0.
EXPERIMENTS = [
    {
        'name': 'MLP_FocalOnly_Gamma2_10pct',
        'use_focal_loss': True,
        'use_weighted_sampler': False,
        'focal_gamma': 2.0,
        'sample_fraction': 0.1,
        'enabled': False,
    },
    {
        'name': 'MLP_SamplerOnly_10pct',
        'use_focal_loss': False,
        'use_weighted_sampler': True,
        'focal_gamma': 2.0,
        'sample_fraction': 0.1,
        'enabled': False,
    },
    {
        'name': 'MLP_FocalOnly_Gamma1_10pct',
        'use_focal_loss': True,
        'use_weighted_sampler': False,
        'focal_gamma': 1.0,
        'sample_fraction': 0.1,
        'enabled': False,
    },
    {
        'name': 'MLP_CE_ClassWeight_10pct',
        'use_focal_loss': False,
        'use_weighted_sampler': False,
        'focal_gamma': 2.0,
        'sample_fraction': 0.1,
        'enabled': False,
    },
    {
        'name': 'MLP_FocalOnly_Gamma05_10pct',
        'use_focal_loss': True,
        'use_weighted_sampler': False,
        'focal_gamma': 0.5,
        'sample_fraction': 0.1,
        'enabled': False,
    },
    {
      'name': 'MLP_CE_ClassWeight_50pct',
      'use_focal_loss': False,
      'use_weighted_sampler': False,
      'focal_gamma': 2.0,
      'sample_fraction': 0.5,
      'enabled': True,
    }
]

if RUN_CICIDS2017_IMBALANCE_EXPERIMENTS:
    import gc
    import pandas as pd
    import torch
    from src.pipeline.dl_runner import run_mlp_experiment_from_output

    rows = []
    for exp in EXPERIMENTS:
        if not exp.get('enabled', False):
            continue
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

        print('Running experiment:', exp['name'])
        report = run_mlp_experiment_from_output(
            output=output,
            dataset_key=CFG['dataset'],
            classification_type=CFG['classification'],
            root=PROJECT_ROOT,
            experiment_name=exp['name'],
            batch_size=CFG['batch_size'],
            epochs=CFG['epochs'],
            lr=1e-3,
            patience=5,
            use_focal_loss=exp['use_focal_loss'],
            use_weighted_sampler=exp['use_weighted_sampler'],
            focal_gamma=exp['focal_gamma'],
            sample_fraction=exp['sample_fraction'],
        )
        metrics = report['models'][exp['name']]
        rows.append({
            'experiment': exp['name'],
            'device': report['device'],
            'sample_fraction': report['sample_fraction'],
            'accuracy': metrics['accuracy'],
            'macro_f1': metrics['macro_f1'],
            'weighted_f1': metrics['weighted_f1'],
            'far': metrics['far'],
            'train_seconds': metrics['train_seconds'],
        })
        print(exp['name'], 'macro_f1=', round(metrics['macro_f1'], 4), 'far=', round(metrics['far'], 4))

    display(pd.DataFrame(rows).sort_values(['far', 'macro_f1'], ascending=[True, False]))
else:
    print('Set RUN_CICIDS2017_IMBALANCE_EXPERIMENTS = True to run selected imbalance experiments.')


## CICIDS2017 Anomaly Detection - Isolation Forest

Train Isolation Forest on Benign-only training rows, tune thresholds on Benign validation rows, and evaluate attack detection on the full test split. This is the first unknown-anomaly branch for the Hybrid NIDS direction.


In [ ]:
RUN_CICIDS2017_ISOLATION_FOREST = True

if RUN_CICIDS2017_ISOLATION_FOREST:
    import pandas as pd
    from src.pipeline.anomaly_runner import run_isolation_forest_experiment

    report = run_isolation_forest_experiment(
        output=output,
        dataset_key=CFG['dataset'],
        classification_type=CFG['classification'],
        root=PROJECT_ROOT,
        experiment_name='IsolationForest_BenignOnly_25pct',
        sample_fraction=0.25,
        target_fars=(0.01, 0.03, 0.05, 0.1),
        n_estimators=200,
    )

    rows = []
    for threshold_name, metrics in report['evaluations'].items():
        row = {'threshold_target': threshold_name}
        row.update(metrics)
        rows.append(row)
    display(pd.DataFrame(rows).sort_values('far'))
else:
    print('Set RUN_CICIDS2017_ISOLATION_FOREST = True to run Isolation Forest anomaly detection.')


## CICIDS2017 Hybrid Decision - RandomForest + Isolation Forest

Evaluate RandomForest as the known-attack classifier and Isolation Forest as the suspicious-unknown detector.


In [ ]:
RUN_CICIDS2017_HYBRID_RF_IF = True

if RUN_CICIDS2017_HYBRID_RF_IF:
    import json
    import joblib
    import numpy as np
    import pandas as pd
    from pathlib import Path
    from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
    from src.evaluation.metrics import compute_classification_metrics

    def _benign_label_index(label_mapping):
        for name, idx in label_mapping.items():
            if str(name).lower() in {'benign', 'normal'}:
                return int(idx)
        return 0

    def _binary_metrics(y_true_attack, y_pred_attack):
        return {
            'accuracy': float(accuracy_score(y_true_attack, y_pred_attack)),
            'precision': float(precision_score(y_true_attack, y_pred_attack, zero_division=0)),
            'recall': float(recall_score(y_true_attack, y_pred_attack, zero_division=0)),
            'f1': float(f1_score(y_true_attack, y_pred_attack, zero_division=0)),
        }

    artifact_dir = PROJECT_ROOT / 'artifacts' / CFG['dataset'] / CFG['classification']
    rf_artifact = artifact_dir / 'RandomForest.pkl'
    anomaly_dir = artifact_dir / 'anomaly'
    if_candidates = sorted(anomaly_dir.glob('IsolationForest_BenignOnly*.pkl'), key=lambda p: p.stat().st_mtime, reverse=True)

    if not rf_artifact.exists():
        raise FileNotFoundError(f'Missing RandomForest artifact: {rf_artifact}')
    if not if_candidates:
        raise FileNotFoundError(f'Missing IsolationForest artifact in: {anomaly_dir}. Run the Isolation Forest cell first.')

    if_artifact = if_candidates[0]
    print('Using RF artifact:', rf_artifact)
    print('Using IF artifact:', if_artifact)

    rf_model = joblib.load(rf_artifact)
    if_bundle = joblib.load(if_artifact)
    if_model = if_bundle['model']
    if_scaler = if_bundle['scaler']
    thresholds = if_bundle['thresholds']

    if_threshold_key = 'far_0.03'
    if if_threshold_key not in thresholds:
        raise KeyError(f'{if_threshold_key} not found. Available: {sorted(thresholds)}')
    threshold = float(thresholds[if_threshold_key])

    # Use a deterministic 10% test sample for quick hybrid evaluation.
    rng = np.random.default_rng(CFG['seed'])
    y_all = np.asarray(output.y_test)
    sample_fraction = 0.1
    sample_idx = []
    for cls in np.unique(y_all):
        cls_idx = np.flatnonzero(y_all == cls)
        n = max(1, int(round(len(cls_idx) * sample_fraction)))
        sample_idx.extend(rng.choice(cls_idx, size=min(n, len(cls_idx)), replace=False).tolist())
    sample_idx = np.array(sorted(sample_idx))

    X_test = output.X_test[sample_idx]
    y_true = y_all[sample_idx]
    benign = _benign_label_index(output.label_mapping)
    y_true_attack = (y_true != benign).astype(int)

    rf_pred = np.asarray(rf_model.predict(X_test))
    rf_attack = (rf_pred != benign).astype(int)

    if_scores = -if_model.decision_function(if_scaler.transform(X_test))
    if_attack = (if_scores >= threshold).astype(int)

    hybrid_attack = ((rf_attack == 1) | ((rf_pred == benign) & (if_attack == 1))).astype(int)
    suspicious_unknown = ((rf_pred == benign) & (if_attack == 1)).astype(int)

    benign_mask = y_true == benign
    attack_mask = ~benign_mask
    rf_missed_attack_mask = attack_mask & (rf_attack == 0)

    rf_multiclass = compute_classification_metrics(y_true, rf_pred, benign_label=benign)
    if_binary = _binary_metrics(y_true_attack, if_attack)
    if_binary.update({
        'far': float(if_attack[benign_mask].mean()) if benign_mask.any() else 0.0,
        'attack_recall': float(if_attack[attack_mask].mean()) if attack_mask.any() else 0.0,
    })
    hybrid_binary = _binary_metrics(y_true_attack, hybrid_attack)
    hybrid_binary.update({
        'far': float(hybrid_attack[benign_mask].mean()) if benign_mask.any() else 0.0,
        'attack_recall': float(hybrid_attack[attack_mask].mean()) if attack_mask.any() else 0.0,
        'suspicious_unknown_alerts': int(suspicious_unknown.sum()),
        'rf_missed_attacks': int(rf_missed_attack_mask.sum()),
        'rf_missed_attacks_caught_by_if': int((rf_missed_attack_mask & (if_attack == 1)).sum()),
        'rf_missed_attack_recovery_rate': float((rf_missed_attack_mask & (if_attack == 1)).sum() / rf_missed_attack_mask.sum()) if rf_missed_attack_mask.any() else 0.0,
    })

    report = {
        'dataset': CFG['dataset'],
        'classification': CFG['classification'],
        'experiment': 'Hybrid_RF_IF_inline_10pct',
        'test_sample_fraction': sample_fraction,
        'rf_artifact': str(rf_artifact.relative_to(PROJECT_ROOT)),
        'if_artifact': str(if_artifact.relative_to(PROJECT_ROOT)),
        'if_threshold_key': if_threshold_key,
        'if_threshold': threshold,
        'n_test': int(len(y_true)),
        'n_test_benign': int(benign_mask.sum()),
        'n_test_attack': int(attack_mask.sum()),
        'rf_multiclass': rf_multiclass,
        'if_binary': if_binary,
        'hybrid_binary': hybrid_binary,
    }
    result_path = PROJECT_ROOT / 'results' / 'cicids2017_multi_Hybrid_RF_IF_inline_10pct_results.json'
    result_path.write_text(json.dumps(report, indent=2), encoding='utf-8')

    rows = [
        {
            'detector': 'RandomForest multiclass',
            'accuracy': rf_multiclass['accuracy'],
            'macro_f1': rf_multiclass['macro_f1'],
            'weighted_f1': rf_multiclass['weighted_f1'],
            'far': rf_multiclass['far'],
            'attack_recall': None,
            'f1': None,
        },
        {'detector': 'IsolationForest anomaly', **if_binary},
        {'detector': 'Hybrid RF OR IF', **hybrid_binary},
    ]
    display(pd.DataFrame(rows))
    print('RF missed attacks:', hybrid_binary['rf_missed_attacks'])
    print('RF missed attacks caught by IF:', hybrid_binary['rf_missed_attacks_caught_by_if'])
    print('Recovery rate:', round(hybrid_binary['rf_missed_attack_recovery_rate'], 4))
    print('Saved:', result_path)
else:
    print('Set RUN_CICIDS2017_HYBRID_RF_IF = True to run hybrid evaluation.')


## 5. Machine-Learning Baselines

These baselines use the new preprocessing path and do not fit the scaler before the train/test split.

In [ ]:
def benign_label_index(label_mapping):
    for name, idx in label_mapping.items():
        if str(name).lower() in {'benign', 'normal'}:
            return int(idx)
    return 0

# CICIDS2017 ML baselines are already saved in results/cicids2017_multi_modular_results.json.
# Keep this disabled by default because LogisticRegression/RandomForest on full CICIDS2017
# can take a long time and may be interrupted in Colab.
RUN_ML_BASELINES = False

all_results = {
    'dataset': CFG['dataset'],
    'classification': CFG['classification'],
    'label_mapping': output.label_mapping,
    'models': {},
}

if RUN_ML_BASELINES:
    ml_models = {
        'LogisticRegression': build_logistic_regression(CFG['seed']),
        'RandomForest': build_random_forest(CFG['seed']),
    }
    for name, model in ml_models.items():
        start = time.time()
        model.fit(output.X_train, output.y_train)
        pred = model.predict(output.X_test)
        metrics = compute_classification_metrics(
            output.y_test,
            pred,
            benign_label=benign_label_index(output.label_mapping),
        )
        metrics['train_seconds'] = time.time() - start
        all_results['models'][name] = metrics
        joblib.dump(model, artifact_dir / f'{name}.pkl')
        print(name, 'macro_f1=', round(metrics['macro_f1'], 4), 'far=', round(metrics['far'], 4))
else:
    existing_path = PROJECT_ROOT / 'results' / f"{CFG['dataset']}_{CFG['classification']}_modular_results.json"
    if existing_path.exists():
        existing = json.loads(existing_path.read_text(encoding='utf-8'))
        for model_name in ['LogisticRegression', 'RandomForest']:
            if model_name in existing.get('models', {}):
                all_results['models'][model_name] = existing['models'][model_name]
        print('ML baselines skipped; loaded existing ML results from:', existing_path)
    else:
        print('ML baselines skipped. Set RUN_ML_BASELINES = True only when you intentionally want to rerun them.')


## 6. Deep-Learning Data Loaders

In [ ]:
def make_loader(X, y, batch_size=512, shuffle=False):
    X = np.asarray(X, dtype=np.float32)
    y = np.asarray(y, dtype=np.int64)
    ds = TensorDataset(torch.tensor(X), torch.tensor(y))
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle)

train_loader = make_loader(output.X_train, output.y_train, CFG['batch_size'], shuffle=True)
val_loader = make_loader(output.X_val, output.y_val, CFG['batch_size'], shuffle=False)
test_loader = make_loader(output.X_test, output.y_test, CFG['batch_size'], shuffle=False)

n_features = output.X_train.shape[1]
n_classes = len(output.label_mapping)
class_weights = compute_balanced_class_weights(output.y_train, device=DEVICE)
print('n_features:', n_features, 'n_classes:', n_classes)


## 7. Deep-Learning Models

The default run trains MLP first because tabular IDS data is easier to debug with an MLP. Add `CNN1D` to `CFG['dl_models']` when the pipeline is stable.

In [ ]:
DL_BUILDERS = {
    'MLP': lambda: MLP(n_features=n_features, n_classes=n_classes),
    'CNN1D': lambda: CNN1D(n_features=n_features, n_classes=n_classes),
}

if CFG['run_dl']:
    for name in CFG['dl_models']:
        model = DL_BUILDERS[name]()
        trainer = TorchTrainer(model, device=DEVICE, lr=1e-3, class_weights=class_weights)
        ckpt_path = artifact_dir / f'{name}.pt'
        start = time.time()
        history = trainer.fit(train_loader, val_loader, epochs=CFG['epochs'], checkpoint_path=ckpt_path, patience=5)
        y_true, y_pred = trainer.predict(test_loader)
        metrics = compute_classification_metrics(y_true, y_pred, benign_label=benign_label_index(output.label_mapping))
        metrics['train_seconds'] = time.time() - start
        metrics['history'] = history
        all_results['models'][name] = metrics
        print(name, 'macro_f1=', round(metrics['macro_f1'], 4), 'far=', round(metrics['far'], 4), 'ckpt=', ckpt_path)
else:
    print('DL training skipped')


## 8. Save Final Report

In [ ]:
results_dir = PROJECT_ROOT / 'results'
results_dir.mkdir(exist_ok=True)
result_path = results_dir / f"{CFG['dataset']}_{CFG['classification']}_modular_results.json"
result_path.write_text(json.dumps(all_results, indent=2), encoding='utf-8')

summary = []
for name, metrics in all_results['models'].items():
    summary.append({
        'model': name,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'weighted_f1': metrics['weighted_f1'],
        'far': metrics['far'],
        'train_seconds': metrics.get('train_seconds'),
    })

print('Saved:', result_path)
pd.DataFrame(summary).sort_values(['macro_f1', 'far'], ascending=[False, True])


## CICIDS2017 Troubleshooting

CICIDS2017 is not downloaded automatically in this notebook. Place the original CSV files under `data/raw/cicids2017/` before selecting `CFG['dataset'] = 'cicids2017'`.

Common issues:

- Empty `data/raw/cicids2017/` raises `No CSV files found`.
- Some CSV mirrors use non-UTF-8 labels; the adapter falls back to `latin1`.
- Some labels contain damaged characters in WebAttack names; label normalization handles this before mapping.
- Some numeric columns contain `Infinity`, `-Infinity`, or string values; preprocessing coerces features to numeric and fills missing values before scaling.
- CICIDS2017 is large. If Colab runs out of RAM, start with fewer CSV files or use a smaller sampled cache first.


## 9. Next Experiment Notes

- Run `nsl_kdd` first for fast debugging.
- Run `unsw_nb15` after NSL-KDD to compare against a more modern dataset.
- For `cicids2017`, place the source CSV files in `data/raw/cicids2017/`; the adapter will merge them and cache `data/cache/cicids2017/merged.csv`.
- This notebook intentionally does not run GAN augmentation yet. The next GAN implementation must follow the SPEC rule: apply GAN only to train-set extreme minority classes.
